### WHO catalogue variant counts

In [10]:
import os
from tqdm import tqdm
import pandas as pd
import numpy as np

In [4]:
# Path to the WHO catalogue csv file
who_cat = "/project/pi_annagreen_umass_edu/saishradha/Data-Curation-for-MTB/dna-tasks/data/WHO_catalogue_2023/WHO_resistance_variants_all_2023.csv"

In [5]:
who_df = pd.read_csv(who_cat)
who_df.head()

,drug,genome_index,confidence,gene,variant,ref_nt,alt_nt
0,Amikacin,2064627.0,4) Not assoc w R - Interim,bacA,bacA_c.102G>A,C,T
1,Amikacin,2063685.0,5) Not assoc w R,bacA,bacA_c.1044G>A,C,T
2,Amikacin,2063685.0,5) Not assoc w R,bacA,bacA_c.1044G>A,CAA,TAG
3,Amikacin,2064624.0,4) Not assoc w R - Interim,bacA,bacA_c.105C>G,G,C
4,Amikacin,2063658.0,4) Not assoc w R - Interim,bacA,bacA_c.1065T>G,TAATCGA,CAACCGC


In [ ]:
# Group by drug and confidence, then count the number of variants for each combination
variant_counts = who_df.groupby(['drug', 'confidence']).size().reset_index(name='variant_count')

# Display the result
print("Variant Counts by Drug and Confidence Category:")
print(variant_counts)


In [ ]:
# Save to CSV (optional)
variant_counts.to_csv("drug_confidence_variant_counts.csv", index=False)

### Variant counts in our genomic dataset

In [11]:
# Directory containing your CSV files
csv_directory = "/project/pi_annagreen_umass_edu/saishradha/project_data_curation/vcf_who_mapped_data"  # Replace with your directory path to the genomic data file containing mapped variants to WHO catalogue
combined_csv_file = "combined_variants.csv"  # Replace with your combined CSV file path

In [12]:
all_dataframes = []
for filename in tqdm(os.listdir(csv_directory), desc="Processing CSVs"):
    if filename.endswith(".csv"):
        file_path = os.path.join(csv_directory, filename)
        df = pd.read_csv(file_path, na_values=[''], keep_default_na=False)
        # Add filename column
        df["Filename"] = filename
        all_dataframes.append(df)

# Combine and export
combined_df = pd.concat(all_dataframes, ignore_index=True)
combined_df.to_csv(combined_csv_file, index=False)


Processing CSVs: 100%|██████████| 17941/17941 [01:56<00:00, 153.83it/s]


In [ ]:
# number of files in the Filename column
num_files = combined_df['Filename'].nunique()
print(f"Number of files processed: {num_files}")

In [13]:
# Specify the drug of interest
drug_of_interest = "Pyrazinamide"  # Replace with your drug of interest
confidence_column = f"{drug_of_interest}_confidence"

In [14]:
# Load the combined CSV file
df = pd.read_csv(combined_csv_file, na_values=[''], keep_default_na=False)

# Initialize a dictionary to store counts for each confidence label
unique_variant_counts = {'R': 0, 'S': 0, 'N/A': 0}

# Drop NaNs in the drug and confidence columns
df = df.dropna(subset=[drug_of_interest, confidence_column])

# Counting unique variants for each confidence category
unique_variant_counts['R'] = df[df[drug_of_interest] == 'R']['variant'].nunique()
unique_variant_counts['S'] = df[df[drug_of_interest] == 'S']['variant'].nunique()
unique_variant_counts['N/A'] = df[df[drug_of_interest] == 'N/A']['variant'].nunique()

# Print the final counts
print(f"Unique Variant Counts for {drug_of_interest} from the Combined CSV:")
print(f"R (Resistant): {unique_variant_counts['R']}")
print(f"S (Sensitive): {unique_variant_counts['S']}")
print(f"N/A (Not Available): {unique_variant_counts['N/A']}")
print(f"Total Unique Variants: {sum(unique_variant_counts.values())}")


Unique Variant Counts for Pyrazinamide from the Combined CSV:
R (Resistant): 292
S (Sensitive): 468
N/A (Not Available): 793
Total Unique Variants: 1553


In [23]:
# Load the combined CSV file
df = pd.read_csv(combined_csv_file, na_values=[''], keep_default_na=False)

# --- 1) Overall % of files with any R call for the drug ---
total_files = df['Filename'].nunique()

# Keep rows that have both the drug label and confidence present
df = df.dropna(subset=[drug_of_interest, confidence_column])

r_df = df[df[drug_of_interest] == 'R']
files_with_any_r = r_df['Filename'].nunique()

overall_pct_files_with_r = (100.0 * files_with_any_r / total_files) if total_files else 0.0
print(f"{files_with_any_r}/{total_files} files ({overall_pct_files_with_r:.2f}%) contain at least one 'R' call for {drug_of_interest}.")


2953/17941 files (16.46%) contain at least one 'R' call for Pyrazinamide.


#### % R variants per drug across all the isolates
1. Rifampicin - 94 (R), 907 (S), 2957 (Total)    (5114/17941 files with R) - 28.51%
2. Isoniazid - 66 (R), 734 (S), 3127 (Total)     (5862/17941 files with R) - 32.67%
3. Ethambutol - 13 (R), 991 (S), 3046 (Total)   (3656/17941 files with R) - 20.38%
4. Ethionamide - 175 (R), 192 (S), 1095 (Total)  (2924/17941 files with R) - 16.29%
5. Pyrazinamide - 292 (R), 468 (S), 1553 (Total)   (2953/17941 files with R) - 16.46%
6. Streptomycin - 117 (R), 276 (S), 1805 (Total)  (4152/17941 files with R) - 23.14%
7. Amikacin - 4 (R), 180 (S), 1083 (Total)    (968/17941 files with R) - 5.39%
8. Kanamycin - 8 (R), 127 (S), 1114 (Total)   (1504/17941 files with R) - 8.39%
9. Levofloxacin - 18 (R), 375 (S), 1198 (Total)  (1516/17941 files with R) - 8.45%
10. Moxifloxacin - 18 (R), 334 (S), 1085 (Total)  (1516/17941 files with R) - 8.45%
11. Capreomycin - 27 (R), 120 (S), 1550 (Total)  (960/17941 files with R) - 5.35%